In [2]:
import pandas as pd

df = pd.read_csv("dataset/synthetic_logs.csv")
df.head(3)

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert


In [3]:
filtered_df = df[df["target_label"] == "Error"]
filtered_df.head()

,timestamp,source,log_message,target_label,complexity
6,3/1/2025 19:14,ModernHR,Shard 6 replication task ended in failure,Error,bert
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert
45,5/22/2025 3:17,ThirdPartyAPI,Data replication task for shard 14 did not com...,Error,bert
51,8/19/2025 5:58,ThirdPartyAPI,Server 4 restarted without warning during data...,Error,bert
55,6/18/2025 11:21,AnalyticsEngine,Server 34 crashed unexpectedly while syncing data,Error,bert


In [4]:
filtered_df = df[df["source"] == "ModernCRM"]
filtered_df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,regex
17,2025-01-27 12:39:05,ModernCRM,nova.osapi_compute.wsgi.server [req-5f1c2027-e...,HTTP Status,bert


In [5]:
df.source.unique()

<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [6]:
df["target_label"].unique()

<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [7]:
df["target_label"].value_counts()

target_label
HTTP Status            1017
Security Alert          371
System Notification     356
Error                   177
Resource Usage          177
Critical Error          161
User Action             144
Workflow Error            4
Deprecation Warning       3
Name: count, dtype: int64

In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

# Generate embeddings for log_message column
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['log_message'].tolist())

embeddings[:2]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([[-1.02939643e-01,  3.35459560e-02, -2.20260546e-02,
         1.55099435e-03, -9.86917224e-03, -1.78956345e-01,
        -6.34410158e-02, -6.01762161e-02,  2.81108748e-02,
         5.99619932e-02, -1.72618870e-02,  1.43367937e-03,
        -1.49560049e-01,  3.15283053e-03, -5.66030554e-02,
         2.71685459e-02, -1.49890343e-02, -3.54037508e-02,
        -3.62936072e-02, -1.45410430e-02, -5.61489770e-03,
         8.75538886e-02,  4.55121212e-02,  2.50963587e-02,
         1.00187939e-02,  1.24266772e-02, -1.39923558e-01,
         7.68696293e-02,  3.14095989e-02, -4.15256852e-03,
         4.36903425e-02,  1.71250161e-02, -8.00951570e-02,
         5.74005991e-02,  1.89092141e-02,  8.55261162e-02,
         3.96399684e-02, -1.34371862e-01, -1.44364685e-03,
         3.06711951e-03,  1.76854104e-01,  4.44885204e-03,
        -1.69274919e-02,  2.24267244e-02, -4.35050242e-02,
         6.09030714e-03, -9.98171326e-03, -6.23972081e-02,
         1.07372720e-02, -6.04900671e-03, -7.14660808e-0

In [10]:
print(embeddings.shape)

(2410, 384)


In [15]:
# Cluster embeddings using DBSCAN
dbscan = DBSCAN(eps=0.1, min_samples=1, metric='cosine')
clusters = dbscan.fit_predict(embeddings)

# Add cluster labels to the dataframe
df['cluster'] = clusters
df.head()

,timestamp,source,log_message,target_label,complexity,cluster
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [16]:
df[df.cluster == 1]

,timestamp,source,log_message,target_label,complexity,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
361,11/19/2025 23:06,BillingSystem,Email service experienced a sending issue,Error,bert,1
1901,9/22/2025 2:52,ThirdPartyAPI,Email service experienced a mail sending issue,Error,bert,1


In [18]:
# Count records per cluster and sort descending
cluster_counts = df['cluster'].value_counts().sort_values(ascending=False)

# Iterate clusters with more than 10 records and print 5 log messages from each
for cluster_id, count in cluster_counts.items():
    if count > 10:
        print(f"\nCluster {cluster_id} (records: {count}):")
        sample_logs = df[df['cluster'] == cluster_id]['log_message'].head(5)
        for log in sample_logs:
            print(log)


Cluster 0 (records: 765):
nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2675118
nova.osapi_compute.wsgi.server [req-4895c258-b2f8-488f-a2a3-4fae63982e48 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" HTTP status code -  200 len: 211 time: 0.0968180
nova.osapi_compute.wsgi.server [req-ee8bc8ba-9265-4280-9215-dbe000a41209 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" RCODE  200 len: 1874 time: 0.2280791
nova.osapi_compute.wsgi.server [req-f0bffbc3-5ab0-4916-91c1-0a61dd7d4ec2 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4

In [19]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message):
            return label
    return None

In [20]:
classify_with_regex("User User123 logged in.")

'User Action'

In [21]:
classify_with_regex("System reboot initiated by user User179.")

'System Notification'

In [22]:
classify_with_regex("Hey there, this is a random log message that doesn't match any pattern.")

In [23]:
# Apply regex classification
df['regex_label'] = df['log_message'].apply(lambda x: classify_with_regex(x))
df[df['regex_label'].notnull()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
7,10/11/2025 8:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,regex,4,System Notification
14,1/4/2025 1:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,regex,4,System Notification
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,regex,9,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,regex,10,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,regex,13,User Action
...,...,...,...,...,...,...,...
2376,6/27/2025 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,regex,27,System Notification
2381,9/5/2025 6:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,regex,45,System Notification
2394,4/3/2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,regex,45,System Notification
2395,5/2/2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,regex,15,System Notification


In [24]:
df[df['regex_label'].isnull()].head(5)

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,NaN
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,NaN
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,NaN
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,NaN
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,NaN


Classification Stage 2: Classification Using Embeddings

In [25]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex.shape

(1910, 7)

In [27]:
label_counts = df_non_regex['target_label'].value_counts()
rare_labels = label_counts[label_counts <= 5].index
print(df_non_regex[df_non_regex['target_label'].isin(rare_labels)]['target_label'].unique())

<StringArray>
['Workflow Error', 'Deprecation Warning']
Length: 2, dtype: str


Classification using LLM for log sources having less than 5 rows or log entries

In [26]:
df_legacy = df_non_regex[df_non_regex.source=="LegacyCRM"]
df_legacy

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
60,2025-10-06 16:55:23,LegacyCRM,Lead conversion failed for prospect ID 7842 du...,Workflow Error,llm,33,NaN
255,2025-05-03 16:55:35,LegacyCRM,API endpoint 'getCustomerDetails' is deprecate...,Deprecation Warning,llm,83,NaN
377,2025-06-24 12:16:29,LegacyCRM,Customer follow-up process for lead ID 5621 fa...,Workflow Error,llm,117,NaN
1325,2025-04-17 07:33:44,LegacyCRM,Escalation rule execution failed for ticket ID...,Workflow Error,llm,311,NaN
1734,2025-04-30 07:47:30,LegacyCRM,The 'ExportToCSV' feature is outdated. Please ...,Deprecation Warning,llm,382,NaN
1826,2025-01-23 10:33:36,LegacyCRM,Support for legacy authentication methods will...,Deprecation Warning,llm,402,NaN
2217,2025-05-12 09:46:54,LegacyCRM,Task assignment for TeamID 3425 could not comp...,Workflow Error,llm,478,NaN


Classification using BERT for log sources having 5 or more than 5 rows or log entries

In [30]:
df_non_legacy = df_non_regex[df_non_regex.source != "LegacyCRM"]
df_non_legacy.source.unique()


<StringArray>
['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem', 'ThirdPartyAPI']
Length: 5, dtype: str

In [31]:
filtered_embeddings = model.encode(df_non_legacy['log_message'].tolist())

filtered_embeddings[:2]

array([[-1.02939643e-01,  3.35459560e-02, -2.20260546e-02,
         1.55099435e-03, -9.86917224e-03, -1.78956345e-01,
        -6.34410158e-02, -6.01762161e-02,  2.81108748e-02,
         5.99619932e-02, -1.72618870e-02,  1.43367937e-03,
        -1.49560049e-01,  3.15283053e-03, -5.66030554e-02,
         2.71685459e-02, -1.49890343e-02, -3.54037508e-02,
        -3.62936072e-02, -1.45410430e-02, -5.61489770e-03,
         8.75538886e-02,  4.55121212e-02,  2.50963587e-02,
         1.00187939e-02,  1.24266772e-02, -1.39923558e-01,
         7.68696293e-02,  3.14095989e-02, -4.15256852e-03,
         4.36903425e-02,  1.71250161e-02, -8.00951570e-02,
         5.74005991e-02,  1.89092141e-02,  8.55261162e-02,
         3.96399684e-02, -1.34371862e-01, -1.44364685e-03,
         3.06711951e-03,  1.76854104e-01,  4.44885204e-03,
        -1.69274919e-02,  2.24267244e-02, -4.35050242e-02,
         6.09030714e-03, -9.98171326e-03, -6.23972081e-02,
         1.07372720e-02, -6.04900671e-03, -7.14660808e-0

In [32]:
filtered_embeddings.shape

(1903, 384)

In [34]:
len(filtered_embeddings)

1903

In [35]:
X = filtered_embeddings
y = df_non_legacy['target_label'].values

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [37]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.989492119089317


In [42]:
import joblib
joblib.dump(clf, '../models/log_classifier.joblib')

['../models/log_classifier.joblib']